<a href="https://colab.research.google.com/github/sergioleite37/exercicio-21/blob/main/XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nova seção

# Nova seção

In [1]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 17.3 MB/s eta 0:00:00


In [2]:
# =============================================================================
# modelo_xgb.py — XGBoost + Optuna + Calibração — Framework v3.0-OPT
# =============================================================================
# Execução independente:  python modelo_xgb.py
# Saídas em resultados/:  xgb_metrics.pkl · xgb_imps.pkl · xgb_calib.pkl
#                         xgb_model.pkl   · xgb_imp_sc.pkl · xgb_ranking.pkl
# =============================================================================

from commons import *  # noqa: F401, F403

from xgboost import XGBClassifier

import matplotlib
matplotlib.use('Agg')

N_OPTUNA_TRIALS_XGB = 50
USE_OPTUNA_XGB      = True


# =============================================================================
# TREINO XGB + OPTUNA + CALIBRAÇÃO (OPT O07: early_stopping consolidado)
# =============================================================================

def train_xgboost_optuna(X_train, y_train, X_val, y_val, X_test, y_test, fold):
    """
    XGBoost com Optuna TPE (v2 C07).
    P03: CalibratedClassifierCV(sigmoid) após treino.
    OPT O07: early_stopping_rounds consolidado no espaço de busca.
    """
    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    spw   = float(n_neg / n_pos) if n_pos > 0 else 1.0

    def _obj(trial):
        p = {
            'n_estimators':        trial.suggest_int('n_estimators', 100, 1000),
            'max_depth':           trial.suggest_int('max_depth', 3, 9),
            'learning_rate':       trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'subsample':           trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree':    trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha':           trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda':          trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'gamma':               trial.suggest_float('gamma', 0, 5),
            'min_child_weight':    trial.suggest_int('min_child_weight', 1, 10),
            'scale_pos_weight':    spw,
            'use_label_encoder':   False,
            'eval_metric':         'logloss',
            'verbosity':           0,
            'random_state':        RANDOM_SEED,
            'n_jobs':              -1,
            'early_stopping_rounds': 30,    # OPT O07: fixo no trial
        }
        m = XGBClassifier(**p)
        m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        return average_precision_score(y_val, m.predict_proba(X_val)[:, 1])

    if USE_OPTUNA_XGB:
        print(f"\n[XGB | Fold {fold}] Optuna ({N_OPTUNA_TRIALS_XGB} trials)...")
        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
        study.optimize(_obj, n_trials=N_OPTUNA_TRIALS_XGB, show_progress_bar=False)
        bp = study.best_params
        bp.update({'scale_pos_weight': spw, 'use_label_encoder': False,
                   'eval_metric': 'logloss', 'verbosity': 0,
                   'random_state': RANDOM_SEED, 'n_jobs': -1,
                   'early_stopping_rounds': 50})
        print(f"[XGB | Fold {fold}] Best PR-AUC(val)={study.best_value:.4f}")
    else:
        bp = {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05,
              'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.1,
              'reg_lambda': 1.0, 'gamma': 1, 'min_child_weight': 3,
              'scale_pos_weight': spw, 'use_label_encoder': False,
              'eval_metric': 'logloss', 'verbosity': 0,
              'random_state': RANDOM_SEED, 'n_jobs': -1,
              'early_stopping_rounds': 50}

    xgb = XGBClassifier(**bp)
    xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    # P03 — Platt Scaling (sigmoid) — mais estável para XGBoost
    probs_val_raw  = xgb.predict_proba(X_val)[:, 1]
    calibrated_xgb = calibrate_tabular(xgb, X_val, y_val, method='sigmoid')
    probs_val      = calibrated_xgb.predict_proba(X_val)[:, 1]
    thr            = optimize_threshold_pr(y_val, probs_val)
    probs_test     = calibrated_xgb.predict_proba(X_test)[:, 1]
    preds_test     = (probs_test >= thr).astype(int)

    metrics  = compute_metrics(y_test, preds_test, probs_test, 'XGBoost', fold)
    feat_imp = pd.Series(xgb.feature_importances_, index=FEATURES_TABULAR)
    print(f"[XGB | Fold {fold}] F1={metrics['F1-Score']:.4f} | "
          f"ROC-AUC={metrics['ROC-AUC']:.4f} | Brier={metrics['Brier Score']:.5f}")

    return (calibrated_xgb, preds_test, probs_test, probs_val_raw, probs_val,
            y_val, metrics, feat_imp, thr)


# =============================================================================
# PIPELINE PRINCIPAL XGB
# =============================================================================

if __name__ == '__main__':
    print("=" * 65)
    print("  MODELO — XGBOOST | Framework v3.0-OPT")
    print("=" * 65)

    # ── Dados ─────────────────────────────────────────────────────────────────
    df_raw     = load_and_preprocess(FILE_PATH)
    df_feat    = create_features(df_raw)
    df_labeled = create_label_for_fold(df_feat, WF_FOLDS[-1][0])
    splits     = get_walk_forward_splits(df_labeled)
    if not splits:
        raise ValueError("Nenhum fold válido. Verifique datas e FILE_PATH.")

    # ── Coletores ─────────────────────────────────────────────────────────────
    all_metrics  = []
    all_imps     = []
    calib_data   = None
    last_model   = None
    last_imp_sc  = None
    last_ranking = None

    # ── Loop Walk-Forward ─────────────────────────────────────────────────────
    for fi, split in enumerate(splits, 1):
        print(f"\n{'═'*65}")
        print(f"  FOLD {fi}/{len(splits)} | ≤{split['train_end']} → ≤{split['test_end']}")
        print(f"{'═'*65}")

        (df_train, df_val, df_test,
         X_tr, X_vl, X_te,
         y_tr, y_vl, y_te, imp, sc) = prepare_fold_data(df_feat, split)

        if len(df_train) < 500 or len(df_test) < 50:
            print(f"[WARN] Fold {fi}: dados insuficientes, pulando")
            continue

        print(f"\n[F{fi}] Treino={len(df_train):,} | Val={len(df_val):,} "
              f"| Teste={len(df_test):,} | Anom.treino={y_tr.mean()*100:.1f}%")

        (xgb_m, _, xgb_prob_te, xgb_pvl_raw, xgb_pvl_cal,
         xgb_y_vl, xgb_met, xgb_imp, _) = train_xgboost_optuna(
            X_tr, y_tr, X_vl, y_vl, X_te, y_te, fi)

        all_metrics.append(xgb_met)
        all_imps.append(xgb_imp)

        if fi == len(splits):
            calib_data   = (xgb_y_vl, xgb_pvl_raw, xgb_pvl_cal)
            last_model   = xgb_m
            last_imp_sc  = (imp, sc)
            last_ranking = rank_single_model(
                df_labeled, xgb_m, 'xgb', 14, imp, sc, 'XGBoost')

    # ── Persistência ──────────────────────────────────────────────────────────
    if all_metrics:
        save_pickle(all_metrics,  'xgb_metrics.pkl')
        save_pickle(all_imps,     'xgb_imps.pkl')
        save_pickle(calib_data,   'xgb_calib.pkl')
        save_pickle(last_model,   'xgb_model.pkl')
        save_pickle(last_imp_sc,  'xgb_imp_sc.pkl')
        save_pickle(last_ranking, 'xgb_ranking.pkl')
        print(f"\n[XGB] Concluído — {len(all_metrics)} folds salvos em resultados/")
    else:
        print("[ERROR] Nenhuma métrica coletada.")

  MODELO — XGBOOST | Framework v3.0-OPT

─────────────────────────────────────────────────────────────────
  FASE 1 — CARREGAMENTO E PRÉ-PROCESSAMENTO
─────────────────────────────────────────────────────────────────
[OK]  102,843 registros | 79 usinas | 2022-01-01 → 2025-12-31
[LABEL] Fold ≤2024-12-31: 1,064 anomalias (1.0%) de 102,764
[WF]  Fold 2022-12-31: treino=25,150 | val=13,213 | teste=13,457
[WF]  Fold 2023-12-31: treino=51,820 | val=13,306 | teste=12,904
[WF]  Fold 2024-12-31: treino=78,030 | val=12,100 | teste=12,634

═════════════════════════════════════════════════════════════════
  FOLD 1/3 | ≤2022-12-31 → ≤2023-12-31
═════════════════════════════════════════════════════════════════
[LABEL] Fold ≤2022-12-31: 1,220 anomalias (1.2%) de 102,764

[F1] Treino=24,625 | Val=13,213 | Teste=13,450 | Anom.treino=0.6%

[XGB | Fold 1] Optuna (50 trials)...
[XGB | Fold 1] Best PR-AUC(val)=0.0799
[THR] Threshold ótimo: 0.0626 (F1_val=0.2222)
[XGB | Fold 1] F1=0.0485 | ROC-AUC=0.7623 | 